In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.interpolate import griddata
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os

In [2]:
mesh_size = 2.0
normal_stress = 5.0
STIFFNESSES = [750, 1050, 1500, 2100, 3000, 4200, 6000, 8400, 12000]

control_mode = 0
comparing_mode = "Stiffness"
CONTROL_MODES = ["NormalStressDC", "NormalStressPC"]
CONTROL_MODES_TITLE = ["Displacement Control", "Stress Control"]

# Rupture positions for animation frames
RUPTURE_POSITIONS = [5, 15, 25, 35, 45, 55, 65, 75, 80, 85, 90, 95, 100, 105, 110, 115]

# Create output folder for GIFs
BASE_FOLDER = f'./Friction-Coefficient'
GIF_FOLDER = os.path.join(BASE_FOLDER, f'{comparing_mode}GIF')
os.makedirs(GIF_FOLDER, exist_ok=True)

# Setup colormap for different stiffnesses
norm = mcolors.Normalize(vmin=min(STIFFNESSES), vmax=max(STIFFNESSES))
colormap = cm.viridis  # You can change to other colormaps like 'plasma', 'coolwarm', etc.
sm = cm.ScalarMappable(norm=norm, cmap=colormap)
sm.set_array([])

# Storage for all data
all_frame_data = []

print("Collecting data for all stiffnesses and positions...")

for RUPTURE_POSITION in RUPTURE_POSITIONS:
    frame_data = {
        'position': RUPTURE_POSITION,
        's12_data': [],
        'mu_data': []
    }
    
    for Spring_stiffness in STIFFNESSES:
        # Construct data path for each stiffness
        DATA_FOLDER = f'./{comparing_mode}/Y{Spring_stiffness}/ShearFace'
        npz_file = os.path.join(DATA_FOLDER, f'ShearFace-{RUPTURE_POSITION}.npz')
        
        if not os.path.exists(npz_file):
            print(f"File {npz_file} not found, skipping...")
            continue
        
        print(f"Loading Stiffness={Spring_stiffness}, Position={RUPTURE_POSITION}...")
        
        # Load data
        data = np.load(npz_file)
        x = data['x']
        y = data['y']
        z = data['z']
        s12 = data['s12']
        mu = data['mu']
        
        # === Process S12 data along midline ===
        # Find midline along z
        z_min, z_max = z.min(), z.max()
        z_mid = 0.5 * (z_min + z_max)
        z_span = z_max - z_min
        
        # Select points near z midline
        tol_z = 0.02 * z_span  # 2% tolerance
        mid_mask_z = np.abs(z - z_mid) <= tol_z
        
        # Bin along y for S12
        y_min, y_max = y.min(), y.max()
        NUM_Y_BINS = 100
        y_edges = np.linspace(y_min, y_max, NUM_Y_BINS + 1)
        y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
        
        s12_midline = np.full(NUM_Y_BINS, np.nan)
        
        if np.count_nonzero(mid_mask_z) > 0:
            counts_s12 = np.zeros(NUM_Y_BINS, dtype=int)
            for yy, ss in zip(y[mid_mask_z], s12[mid_mask_z]):
                if not np.isnan(ss):
                    bi = np.searchsorted(y_edges, yy, side='right') - 1
                    if 0 <= bi < NUM_Y_BINS:
                        if np.isnan(s12_midline[bi]):
                            s12_midline[bi] = 0.0
                        s12_midline[bi] += ss
                        counts_s12[bi] += 1
            good_s12 = counts_s12 > 0
            s12_midline[good_s12] /= counts_s12[good_s12]
        
        # === Process mu data along midline ===
        mu_midline = np.full(NUM_Y_BINS, np.nan)
        
        if np.count_nonzero(mid_mask_z) > 0:
            counts_mu = np.zeros(NUM_Y_BINS, dtype=int)
            for yy, mm in zip(y[mid_mask_z], mu[mid_mask_z]):
                if not np.isnan(mm):
                    bi = np.searchsorted(y_edges, yy, side='right') - 1
                    if 0 <= bi < NUM_Y_BINS:
                        if np.isnan(mu_midline[bi]):
                            mu_midline[bi] = 0.0
                        mu_midline[bi] += mm
                        counts_mu[bi] += 1
            good_mu = counts_mu > 0
            mu_midline[good_mu] /= counts_mu[good_mu]
        
        # Store processed data
        frame_data['s12_data'].append({
            'stiffness': Spring_stiffness,
            'y_centers': y_centers,
            's12_midline': s12_midline,
            'color': colormap(norm(Spring_stiffness))
        })
        
        frame_data['mu_data'].append({
            'stiffness': Spring_stiffness,
            'y_centers': y_centers,
            'mu_midline': mu_midline,
            'color': colormap(norm(Spring_stiffness))
        })
    
    if frame_data['s12_data'] and frame_data['mu_data']:
        all_frame_data.append(frame_data)

# === Create combined animation ===
print("\nCreating combined S12 and Mu animation...")

if all_frame_data:
    # Find global limits for consistent axes
    all_s12_values = []
    all_mu_values = []
    y_global_min, y_global_max = float('inf'), float('-inf')
    
    for frame in all_frame_data:
        for s12_data in frame['s12_data']:
            valid_s12 = s12_data['s12_midline'][~np.isnan(s12_data['s12_midline'])]
            if len(valid_s12) > 0:
                all_s12_values.extend(valid_s12)
            y_global_min = min(y_global_min, s12_data['y_centers'].min())
            y_global_max = max(y_global_max, s12_data['y_centers'].max())
        
        for mu_data in frame['mu_data']:
            valid_mu = mu_data['mu_midline'][~np.isnan(mu_data['mu_midline'])]
            if len(valid_mu) > 0:
                all_mu_values.extend(valid_mu)
    
    # Set y-axis limits with some padding
    if all_s12_values:
        s12_min, s12_max = np.percentile(all_s12_values, [1, 99])
        s12_range = s12_max - s12_min
        s12_min -= 0.1 * s12_range
        s12_max += 0.1 * s12_range
    else:
        s12_min, s12_max = -1, 1
    
    if all_mu_values:
        mu_min, mu_max = np.percentile(all_mu_values, [1, 99])
        mu_range = mu_max - mu_min
        mu_min -= 0.1 * mu_range
        mu_max += 0.1 * mu_range
    else:
        mu_min, mu_max = 0, 1
    
    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), constrained_layout=True)
    
    # Add colorbar
    cbar = fig.colorbar(sm, ax=[ax1, ax2], aspect=30, pad=0.02)
    cbar.set_label('Stiffness (MPa)', rotation=270, labelpad=20)
    
    def animate(frame_idx):
        ax1.clear()
        ax2.clear()
        
        frame_data = all_frame_data[frame_idx]
        position = frame_data['position']
        
        # Plot S12 data (upper subplot)
        for s12_data in frame_data['s12_data']:
            valid_mask = ~np.isnan(s12_data['s12_midline'])
            if np.any(valid_mask):
                ax1.plot(s12_data['y_centers'][valid_mask], 
                        s12_data['s12_midline'][valid_mask],
                        'o-', linewidth=2, markersize=3,
                        color=s12_data['color'],
                        label=f"E={s12_data['stiffness']}MPa")
        
        ax1.set_xlabel('Y Position (mm)')
        ax1.set_ylabel('Shear Stress S12 (MPa)')
        ax1.set_title(f'S12 at Midline - Position {position}mm, {CONTROL_MODES_TITLE[control_mode]}, σn={normal_stress}MPa')
        ax1.set_xlim(y_global_min, y_global_max)
        ax1.set_ylim(s12_min, s12_max)
        ax1.grid(True, alpha=0.3)
        ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        
        # Plot Mu data (lower subplot)
        for mu_data in frame_data['mu_data']:
            valid_mask = ~np.isnan(mu_data['mu_midline'])
            if np.any(valid_mask):
                ax2.plot(mu_data['y_centers'][valid_mask], 
                        mu_data['mu_midline'][valid_mask],
                        'o-', linewidth=2, markersize=3,
                        color=mu_data['color'],
                        label=f"E={mu_data['stiffness']}MPa")
        
        # Add reference line for mu
        ax2.axhline(y=0.7, color='red', linestyle='--', linewidth=1.5, label='μ=0.7')
        
        ax2.set_xlabel('Y Position (mm)')
        ax2.set_ylabel('Friction Coefficient μ')
        ax2.set_title(f'Friction Coefficient at Midline - Position {position}mm')
        ax2.set_xlim(y_global_min, y_global_max)
        ax2.set_ylim(mu_min, mu_max)
        ax2.grid(True, alpha=0.3)
        
        # Add overall title
        fig.suptitle(f'Rupture Position: {position}mm | Mesh: {mesh_size}mm', 
                    fontsize=14, fontweight='bold')
        
        return ax1.lines + ax2.lines
    
    # Create animation
    anim = FuncAnimation(fig, animate, frames=len(all_frame_data), 
                        interval=500, blit=False, repeat=True)
    
    # Save animation
    gif_path = os.path.join(GIF_FOLDER, 'Stiffness-S12_Mu.gif')
    anim.save(gif_path, writer=PillowWriter(fps=2), dpi=150)
    plt.close(fig)
    
    print(f"Animation saved: {gif_path}")
    print(f"Processed {len(all_frame_data)} frames with {len(STIFFNESSES)} stiffness values")
else:
    print("No data available for animation")

print("\nAnimation creation complete!")

Loading Stiffness=750, Position=5...
Loading Stiffness=1050, Position=5...
Loading Stiffness=1500, Position=5...
Loading Stiffness=2100, Position=5...
Loading Stiffness=3000, Position=5...
Loading Stiffness=4200, Position=5...
Loading Stiffness=6000, Position=5...
Loading Stiffness=8400, Position=5...
Loading Stiffness=12000, Position=5...
Loading Stiffness=750, Position=15...
Loading Stiffness=1050, Position=15...
Loading Stiffness=1500, Position=15...
Loading Stiffness=2100, Position=15...
Loading Stiffness=3000, Position=15...
Loading Stiffness=4200, Position=15...
Loading Stiffness=6000, Position=15...
Loading Stiffness=8400, Position=15...
Loading Stiffness=12000, Position=15...
Loading Stiffness=750, Position=25...
Loading Stiffness=1050, Position=25...
Loading Stiffness=1500, Position=25...
Loading Stiffness=2100, Position=25...
Loading Stiffness=3000, Position=25...
Loading Stiffness=4200, Position=25...
Loading Stiffness=6000, Position=25...
Loading Stiffness=8400, Position=25